In [1]:
import sys
sys.path.append('../')

import torch
import pandas as pd
import optuna
import warnings
from copy import deepcopy
from torch.utils.data import DataLoader
from src.mypackage.data_preparation import prepare_statistical_data
from src.mypackage.torch_dataset import EnergyDataset
from src.mypackage.trainer import Trainer
from src.mypackage.rnn_models import RNNWithCNNModel
from src.mypackage.forecasting import dnn_forecast
from src.mypackage.evaluation import print_metrics, get_true_values
from src.mypackage.visualization import plot_forecast
from src.mypackage.utils import set_seed, SEED

warnings.filterwarnings("ignore")
set_seed(SEED)

In [2]:
# ========== データ読み込みとデータセット準備 ==========
df = pd.read_csv("../data/raw/PJME_hourly.csv")
_, tmp = prepare_statistical_data(df)
timesteps = tmp["Datetime"].copy()
del tmp
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

SEQ_LEN = 168
PRED_LEN = 24
SHIFT = 24
BATCH_SIZE = 32

# ========== データセット作成 ==========
dataset = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=PRED_LEN, mode="re-train")
retrain_dataset = deepcopy(dataset)
train_dataset = deepcopy(dataset)
train_dataset.mode_switch("train")
valid_dataset = deepcopy(dataset)
valid_dataset.mode_switch("val")
test_dataset = deepcopy(dataset)
test_dataset.mode_switch("test")

# DataLoader作成
retrain_loader = DataLoader(retrain_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Whole dataset size: {len(retrain_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Input features: {train_dataset[0][0].shape[1]}")


Dataset shape: (145366, 2)
Columns: ['Datetime', 'PJME_MW']
Whole dataset size: 5654
Train dataset size: 5289
Val dataset size: 358
Test dataset size: 358
Input features: 19


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== ハイパーパラメータ設定 ==========
# 固定
NUM_EPOCHS = 100
INPUT_SIZE = train_dataset[0][0].shape[1]
EARLY_STOPPING_PATIENCE = 10
OUTPUT_SIZE = PRED_LEN

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    hidden_size = 2 ** trial.suggest_int("hidden_size", 4, 9)
    kernel_size = trial.suggest_int("kernel_size", 3, 7)
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 3)
    num_rnn_layers = trial.suggest_int("num_rnn_layers", 1, 4)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    model_type = trial.suggest_categorical("model_type", ["RNN", "LSTM", "GRU"])

    model = RNNWithCNNModel(input_size=INPUT_SIZE, 
                        hidden_size=hidden_size, 
                        kernel_size=kernel_size,
                        num_conv_layers=num_conv_layers,
                        num_layers=num_rnn_layers, 
                        output_size=OUTPUT_SIZE, 
                        dropout=dropout,
                        rnn_type=model_type).to(device)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        checkpoint_dir="../models/rnn_with_cnn_checkpoints"
    )
    history = trainer.train(num_epochs=NUM_EPOCHS, verbose=0)
    best_iteration = trainer.early_stopping.best_iteration
    trial.set_user_attr("best_iteration", best_iteration)
    return min(history['val_loss'])

Using device: cpu


In [ ]:
# パラメータ探索の実行
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=50)
print(study.best_params)

[I 2026-07-10 14:19:37,171] A new study created in memory with name: no-name-f5c94be5-7462-4886-a8ab-57d2ae3f9537


Validation loss decreased (inf --> 0.143679). Saving model...
Validation loss decreased (0.143679 --> 0.138667). Saving model...
EarlyStopping counter: 1 out of 5
EarlyStopping counter: 2 out of 5
Validation loss decreased (0.138667 --> 0.116229). Saving model...
Validation loss decreased (0.116229 --> 0.110563). Saving model...
EarlyStopping counter: 1 out of 5
Validation loss decreased (0.110563 --> 0.104848). Saving model...
EarlyStopping counter: 1 out of 5
EarlyStopping counter: 2 out of 5
EarlyStopping counter: 3 out of 5
EarlyStopping counter: 4 out of 5
EarlyStopping counter: 5 out of 5
Checkpoint loaded from ..\models\rnn_checkpoints\best_model.pt


[I 2026-07-10 15:01:04,744] Trial 0 finished with value: 0.10484765004366636 and parameters: {'learning_rate': 0.0001329291894316216, 'weight_decay': 0.0007114476009343421, 'hidden_size': 8, 'num_layers': 3, 'dropout': 0.1624074561769746, 'model_type': 'GRU'}. Best is trial 0 with value: 0.10484765004366636.


Validation loss decreased (inf --> 0.514089). Saving model...
Validation loss decreased (0.514089 --> 0.348449). Saving model...
Validation loss decreased (0.348449 --> 0.280157). Saving model...
Validation loss decreased (0.280157 --> 0.262270). Saving model...
Validation loss decreased (0.262270 --> 0.229689). Saving model...
Validation loss decreased (0.229689 --> 0.185582). Saving model...
Validation loss decreased (0.185582 --> 0.171473). Saving model...
Validation loss decreased (0.171473 --> 0.157851). Saving model...
Validation loss decreased (0.157851 --> 0.157757). Saving model...
Validation loss decreased (0.157757 --> 0.149983). Saving model...
Validation loss decreased (0.149983 --> 0.139887). Saving model...
EarlyStopping counter: 1 out of 5
EarlyStopping counter: 2 out of 5
Validation loss decreased (0.139887 --> 0.129039). Saving model...
EarlyStopping counter: 1 out of 5
EarlyStopping counter: 2 out of 5
EarlyStopping counter: 3 out of 5
EarlyStopping counter: 4 out of

[I 2026-07-10 15:09:59,857] Trial 1 finished with value: 0.12903850649793944 and parameters: {'learning_rate': 0.0006358358856676254, 'weight_decay': 0.000133112160807369, 'hidden_size': 4, 'num_layers': 4, 'dropout': 0.4329770563201687, 'model_type': 'RNN'}. Best is trial 0 with value: 0.10484765004366636.


EarlyStopping counter: 5 out of 5
Checkpoint loaded from ..\models\rnn_checkpoints\best_model.pt
Validation loss decreased (inf --> 0.426458). Saving model...
Validation loss decreased (0.426458 --> 0.258323). Saving model...
Validation loss decreased (0.258323 --> 0.218681). Saving model...
Validation loss decreased (0.218681 --> 0.192188). Saving model...
Validation loss decreased (0.192188 --> 0.166728). Saving model...
Validation loss decreased (0.166728 --> 0.152248). Saving model...
Validation loss decreased (0.152248 --> 0.150685). Saving model...
Validation loss decreased (0.150685 --> 0.148594). Saving model...


[W 2026-07-10 15:17:12,027] Trial 2 failed with parameters: {'learning_rate': 8.17949947521167e-05, 'weight_decay': 3.752055855124284e-05, 'hidden_size': 6, 'num_layers': 2, 'dropout': 0.34474115788895177, 'model_type': 'GRU'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\sh204\py_env\ds_env\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\sh204\AppData\Local\Temp\ipykernel_5564\936869292.py", line 37, in objective
    history = trainer.train(num_epochs=NUM_EPOCHS, verbose=False)
  File "c:\Users\sh204\portfolio\hourly_energy_consumption\notebooks\..\src\mypackage\trainer.py", line 179, in train
    train_loss, train_mae, train_rmse = self._train_epoch()
                                        ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\sh204\portfolio\hourly_energy_consumption\notebooks\..\src\mypackage\trainer.py", line 128, in _train_epoch
    loss.backward()
 

KeyboardInterrupt: 

In [ ]:
# モデルの再学習（1日予測）
model = RNNWithCNNModel(input_size=INPUT_SIZE,
                        hidden_size=2**study.best_params["hidden_size"], 
                        kernel_size=study.best_params["kernel_size"],
                        num_conv_layers=study.best_params["num_conv_layers"],
                        num_rnn_layers=study.best_params["num_rnn_layers"], 
                        output_size=OUTPUT_SIZE, 
                        dropout=study.best_params["dropout"],
                        rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=retrain_loader,
                  test_loader=test_loader,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_with_cnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_with_cnn_checkpoints/day_best_model.pth")
trainer.save_config("../models/rnn_with_cnn_checkpoints/day_best_model_config.json")


Test Metrics:
  Test Loss (MSE): 0.093888
  Test MAE: 0.206596
  Test RMSE: 0.291224
Checkpoint saved to ..\models\rnn_checkpoints\model_20260508_160620.pt
Configuration saved to ..\models\rnn_checkpoints\training_config_20260508_160621.json


In [ ]:
# 1週間予測用のデータセット
dataset_week = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=7*PRED_LEN, mode="re-train")
test_dataset_week = deepcopy(dataset_week)
test_dataset_week.mode_switch("test")

train_loader_week = DataLoader(dataset_week, batch_size=BATCH_SIZE, shuffle=True)
test_loader_week = DataLoader(test_dataset_week, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# モデルの再学習（1週間予測）
model = RNNWithCNNModel(input_size=INPUT_SIZE,
                        hidden_size=2**study.best_params["hidden_size"], 
                        kernel_size=study.best_params["kernel_size"],
                        num_conv_layers=study.best_params["num_conv_layers"],
                        num_rnn_layers=study.best_params["num_rnn_layers"], 
                        output_size=7*PRED_LEN, 
                        dropout=study.best_params["dropout"],
                        rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_week,
                  test_loader=test_loader_week,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_with_cnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_with_cnn_checkpoints/week_best_model.pth")
trainer.save_config("../models/rnn_with_cnn_checkpoints/week_best_model_config.json")

In [ ]:
# 1か月予測用のデータセット
dataset_month = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=30*PRED_LEN, mode="re-train")
test_dataset_month = deepcopy(dataset_month)
test_dataset_month.mode_switch("test")

train_loader_month = DataLoader(dataset_month, batch_size=BATCH_SIZE, shuffle=True)
test_loader_month = DataLoader(test_dataset_month, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# モデルの再学習（1か月予測）
model = RNNWithCNNModel(input_size=INPUT_SIZE,
                        hidden_size=2**study.best_params["hidden_size"], 
                        kernel_size=study.best_params["kernel_size"],
                        num_conv_layers=study.best_params["num_conv_layers"], 
                        num_rnn_layers=study.best_params["num_rnn_layers"], 
                        output_size=30*PRED_LEN,  # 1か月予測のため出力サイズを変更
                        dropout=study.best_params["dropout"],
                        rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_month,
                  test_loader=test_loader_month,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_with_cnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_with_cnn_checkpoints/month_best_model.pth")
trainer.save_config("../models/rnn_with_cnn_checkpoints/month_best_model_config.json")

In [ ]:
# 1年予測用のデータセット
dataset_year = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=365*PRED_LEN, mode="re-train")
test_dataset_year = deepcopy(dataset_year)
test_dataset_year.mode_switch("test")

train_loader_year = DataLoader(dataset_year, batch_size=BATCH_SIZE, shuffle=True)
test_loader_year = DataLoader(test_dataset_year, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# モデルの再学習（1年予測）
model = RNNWithCNNModel(input_size=INPUT_SIZE,
                        hidden_size=2**study.best_params["hidden_size"], 
                        kernel_size=study.best_params["kernel_size"],
                        num_conv_layers=study.best_params["num_conv_layers"], 
                        num_rnn_layers=study.best_params["num_rnn_layers"], 
                        output_size=365*PRED_LEN,  # 1年予測のため出力サイズを変更
                        dropout=study.best_params["dropout"],
                        rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_year,
                  test_loader=test_loader_year,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_with_cnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_with_cnn_checkpoints/year_best_model.pth")
trainer.save_config("../models/rnn_with_cnn_checkpoints/year_best_model_config.json")

In [ ]:
# チェックポイントの読み込みと再学習
checkpoint_day = torch.load("../models/rnn_with_cnn_checkpoints/day_best_model.pth", map_location=device)
params_day = checkpoint_day["model_config"]
model_day = RNNWithCNNModel(**params_day).to(device)
model_day.load_state_dict(checkpoint_day["model_state_dict"])

checkpoint_week = torch.load("../models/rnn_with_cnn_checkpoints/week_best_model.pth", map_location=device)
params_week = checkpoint_week["model_config"]
model_week = RNNWithCNNModel(**params_week).to(device)
model_week.load_state_dict(checkpoint_week["model_state_dict"])

checkpoint_month = torch.load("../models/rnn_with_cnn_checkpoints/month_best_model.pth", map_location=device)
params_month = checkpoint_month["model_config"]
model_month = RNNWithCNNModel(**params_month).to(device)
model_month.load_state_dict(checkpoint_month["model_state_dict"])

checkpoint_year = torch.load("../models/rnn_with_cnn_checkpoints/year_best_model.pth", map_location=device)
params_year = checkpoint_year["model_config"]
model_year = RNNWithCNNModel(**params_year).to(device)
model_year.load_state_dict(checkpoint_year["model_state_dict"])

In [ ]:
y_pred_day = dnn_forecast(model_day, test_loader, device)
y_pred_week = dnn_forecast(model_week, test_loader_week, device)
y_pred_month = dnn_forecast(model_month, test_loader_month, device)
y_pred_year = dnn_forecast(model_year, test_loader_year, device)

y_true = get_true_values(test_loader)

In [ ]:
print_metrics(y_true, y_pred_day, "RNN Forecast (1 Day)")
print_metrics(y_true, y_pred_week, "RNN Forecast (1 Week)")
print_metrics(y_true, y_pred_month, "RNN Forecast (1 Month)")
print_metrics(y_true, y_pred_year, "RNN Forecast (1 Year)")

In [ ]:
plot_forecast(y_true, y_pred_day, timesteps, "RNN and CNN Forecast (1 Day)")

In [ ]:
plot_forecast(y_true, y_pred_week, timesteps, "RNN and CNN Forecast (1 Week)")

In [ ]:
plot_forecast(y_true, y_pred_month, timesteps, "RNN and CNN Forecast (1 Month)")

In [ ]:
plot_forecast(y_true, y_pred_year, timesteps, "RNN and CNN Forecast (1 Year)")